# 04C KV Cache 与 Prompt 排列：模型实际算了什么

前两份实践检查“模型看见什么”和“记录怎样保存”。这一份沿课件 L04.02-S03 至 S06，回答三个更底层的问题：预填充和逐 Token 解码怎样复用 Key/Value；两种相同内容的 Prompt 排列怎样改变共同前缀；为什么共同前缀不等于服务端缓存命中。

证据来自**已训练的 Qwen2.5-0.5B-Instruct 本地模型**的真实前向调用，不是随机初始化的玩具模型。模型版本、张量形状、输出和两种排列的实际回答已保存到结果文件。默认读取保存结果，不下载权重、不调用 API；显式开启本地复现时才加载模型。服务端实验另用同一 DeepSeek 模型连续发送两组 Prompt 排列，逐次读取缓存字段；两种模型的 Token 数不能混算。


## 学习路线与配套课件

实践 4C：KV Cache 与 Prompt 排列（`L04-P03`）。

建议先完成第四节概念正课，再看本实验。对应课件稳定编号：L04.02-S03、L04.02-S04、L04.02-S05、L04.02-S06。

按“真实模型证据—前缀对照—容量边界—练习—审计报告”顺序学习。默认播放已保存的真实结果；要复现模型前向计算，先将固定版本模型放入本地缓存，再显式打开开关。


In [1]:
from pathlib import Path
import json, hashlib
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "04 上下文工程与记忆"
record = json.loads((CHAPTER / "数据/实验记录/local-kv-prompt.json").read_text(encoding="utf-8"))
assert record["kind"] == "real_local_pretrained_model"
assert hashlib.sha256(record["shared_material"].encode()).hexdigest() == record["shared_material_sha256"]
print("真实本地模型:", record["model"])
print("固定版本:", record["revision"])
print("运行条件:", record["device"], record["dtype"], "Transformers", record["transformers_version"])


真实本地模型: Qwen/Qwen2.5-0.5B-Instruct
固定版本: 7ae557604adf67be50417f59c2c2f167def9a775
运行条件: cpu float32 Transformers 4.56.2


## 1. 预填充与增量解码复用什么

预填充处理整段输入，得到本轮下一 Token 的分布，同时在每层留下历史位置的 Key 和 Value。后续解码只输入尚未处理的新 Token，并把它的 Key/Value 追加到缓存。这里用**同一个已训练模型**验证：一次完整前向与“前面先预填充、最后一个 Token 增量处理”的末位 logits 应在浮点误差内一致。

下面读取真实前向记录，核对层数、张量形状、序列长度、缓存字节数和最大差异。缓存字节数是这个 float32 本地模型在这一轮的 K/V 张量大小；它不包括模型权重或推理服务的其他内存。


In [2]:
kv = record["kv"]
print("输入 Token:", kv["prompt_tokens"], "层数:", kv["layers"])
print("首层 Key/Value 形状:", kv["key_shape_first_layer"], kv["value_shape_first_layer"])
print("缓存序列长度:", kv["cached_sequence_tokens"])
print("K/V 张量 MiB:", round(kv["cache_bytes_float32"] / 1024**2, 3))
print("完整与增量前向的最大 logit 差:", kv["max_abs_logit_difference"])
assert kv["cached_sequence_tokens"] == kv["prompt_tokens"]
assert kv["max_abs_logit_difference"] < 1e-3
assert kv["key_shape_first_layer"] == kv["value_shape_first_layer"]


输入 Token: 69 层数: 24
首层 Key/Value 形状: [1, 2, 69, 64] [1, 2, 69, 64]
缓存序列长度: 69
K/V 张量 MiB: 1.617
完整与增量前向的最大 logit 差: 3.528594970703125e-05


## 2. 可以复现的真实模型调用

下格保留完整的模型加载和两种前向代码。默认开关关闭，录制时读上面的已保存输出；若本机已经有固定版本模型，可显式开启，并在新内核比较本次结果。加载器被限制为只读本地缓存，不会静默联网下载。

这里只检查数值等价与计算状态复用。KV Cache 不决定哪些历史值得保留，也不会自动把上一次**独立请求**没有发送的内容变成这次的上下文。


In [3]:
RUN_LOCAL_MODEL = False  # 本地已有固定版本模型时，才显式改为 True。
if RUN_LOCAL_MODEL:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        record["model"], revision=record["revision"],
        local_files_only=True, trust_remote_code=False)
    model = AutoModelForCausalLM.from_pretrained(
        record["model"], revision=record["revision"],
        local_files_only=True, trust_remote_code=False, dtype=torch.float32).eval()
    ids = tokenizer.apply_chat_template(
        record["layouts"]["stable_first"]["messages_a"],
        add_generation_prompt=True, return_tensors="pt")
    with torch.inference_mode():
        full = model(ids, use_cache=False).logits[:, -1, :]
        prefill = model(ids[:, :-1], use_cache=True)
        decoded = model(ids[:, -1:], past_key_values=prefill.past_key_values,
                        use_cache=True)
    difference = float((full - decoded.logits[:, -1, :]).abs().max())
    print("本次最大 logit 差:", difference)
    print("本次缓存 Token:", decoded.past_key_values.get_seq_length())
    assert difference < 1e-3
else:
    print("读取已保存的真实模型前向结果；本格没有加载权重或发出 API 请求。")


读取已保存的真实模型前向结果；本格没有加载权重或发出 API 请求。


## 3. 同样的内容，Prompt 排列改变了什么

我们固定系统规则、资料、问题和模型，只交换用户消息内部“共同资料”和“当前问题”的先后位置。两次请求的共同前缀长度是逐 Token 比较得到的，不是按字符猜测。把稳定规则与资料放在前面，通常会让变化的后缀更晚出现；把当前问题放在前面，共同前缀会更早中断。

这次模型在两种排列下都回答了同一个文件名。**共同前缀更长不证明答案更正确**；它只提供一种可能的跨请求复用条件。还要看模型回答与服务端真正返回的缓存统计。


In [4]:
layouts = record["layouts"]
def common_prefix(left, right):
    return next((i for i, (a, b) in enumerate(zip(left, right)) if a != b),
                min(len(left), len(right)))
for name, item in layouts.items():
    measured = common_prefix(item["tokens_a"], item["tokens_b"])
    assert measured == item["shared_prefix_tokens"]
    print(name, "两次请求共同前缀 Token:", measured,
          "输入长度:", len(item["tokens_a"]), len(item["tokens_b"]))
    print("同一问题的模型回答:", item["answer_a"])
assert layouts["stable_first"]["shared_prefix_tokens"] > layouts["question_first"]["shared_prefix_tokens"]


stable_first 两次请求共同前缀 Token: 52 输入长度: 69 73
同一问题的模型回答: progress.json
question_first 两次请求共同前缀 Token: 28 输入长度: 69 73
同一问题的模型回答: progress.json


### 把真实 KV 张量换算成并发容量

配套课件：L04.02-S04、L04.02-S05、L04.02-S06。

上面不是画了一个缓存概念图，而是读到了预训练模型在一次前向计算后返回的逐层 K/V 张量。现在以这次实测的每 Token 字节数做容量估算；同时把两种 Prompt 排列的共同前缀长度放在旁边。这里的字节数是同一模型、同一精度下的近似线性外推，不是生产推理服务的显存上限。


In [5]:
bytes_per_token = kv["cache_bytes_float32"] / kv["prompt_tokens"]
estimated_single = round(bytes_per_token * 4096 / 1024**2, 1)
estimated_eight = round(bytes_per_token * 4096 * 8 / 1024**2, 1)
print("本模型实测每 Token KV 字节:", round(bytes_per_token))
print("4096 Token 单请求估算 MiB:", estimated_single)
print("八个同长度请求估算 MiB:", estimated_eight)
print("两次请求的共同前缀:", {name: item["shared_prefix_tokens"] for name, item in layouts.items()})
assert abs(bytes_per_token - 24 * 2 * 2 * 64 * 4) < 1
assert layouts["stable_first"]["shared_prefix_tokens"] > layouts["question_first"]["shared_prefix_tokens"]


本模型实测每 Token KV 字节: 24576
4096 Token 单请求估算 MiB: 96.0
八个同长度请求估算 MiB: 768.0
两次请求的共同前缀: {'stable_first': 52, 'question_first': 28}


**结果解读**

实测缓存覆盖 24 层，每层两组 Key/Value，序列长 69。这个 float32 例子每 Token 约占 24,576 字节；外推到 4096 Token、八个并发请求时约占 768 MiB。实际服务还可能使用低精度、分页、共享或淘汰；不同服务的缓存命中与保留时间不能从这张表推断。

**小练习**

把并发数从八改成十六，先预测 KV 容量估算怎样变化。再解释为什么稳定前缀的 52 个共同 Token 不等于服务端已经命中了 52 个缓存 Token。

<details><summary>完成后展开参考分析</summary>

在同一线性估算前提下，并发翻倍，容量估算也翻倍；共同 Token 只说明输入前缀相同，是否跨请求复用还取决于服务实现、阈值、生命周期和实际返回的缓存统计。

</details>


## 4. Prompt Cache 命中要读服务端字段

上一格的共同前缀由本地 Qwen tokenizer 算出，只说明这两条请求在这个模型中的输入前缀相同。跨请求的 Prompt Cache 是否启用、阈值、保留时间和计费都由服务商决定，不能把“共同前缀 52 个 Token”说成“命中 52 个 Token”。

这里读取本实践单独采集的 DeepSeek 对照：每种排列连续提出三个不同问题，系统规则、实验批次和资料固定，只改变“问题”和“资料”的位置。第三次更有机会复用前两次形成的共同前缀，但服务端缓存按实际返回字段计量，命中并不保证。两组请求仍是先后运行，不能仅凭一次试验断言排列必然造成命中差异。保存记录包含六条原始请求、回答、模型与服务用量，可运行 `python scripts/record_context_cache_pairs.py` 显式重新采集；脚本从环境变量读取密钥，不在 Notebook 中自动请求。

本地 Qwen 的共同前缀长度与这组 DeepSeek 请求的命中 Token 属于**不同模型、不同 Prompt**，不能直接比较数字。服务端的前缀持久化与尽力缓存规则参见 [DeepSeek 官方缓存说明](https://api-docs.deepseek.com/guides/kv_cache/)。


In [6]:
prefix_path = CHAPTER / "数据/实验记录/deepseek-prefix-cache.json"
cache_rows = []
if prefix_path.exists():
    deepseek_prefix = json.loads(prefix_path.read_text(encoding="utf-8"))
    assert deepseek_prefix["kind"] == "real_deepseek_prefix_cache_comparison"
    assert len(deepseek_prefix["material_sha256"]) == 64
    print("采集时的原始课程文档哈希:", deepseek_prefix["material_sha256"], "（原文未包含在学生资料中）")
    for name, requests in deepseek_prefix["layouts"].items():
        assert len(requests) == 3
        for item in requests:
            usage = item["usage"]
            hit, miss = usage["prompt_cache_hit_tokens"], usage["prompt_cache_miss_tokens"]
            assert hit + miss == usage["prompt_tokens"]
            cache_rows.append({"排列": name, "轮次": item["number"],
                               "输入Token": usage["prompt_tokens"],
                               "命中Token": hit, "未命中Token": miss})
            print(cache_rows[-1])
    assert len(cache_rows) == 6
    print("同一排列内比较第 1、2、3 次；服务端命中不保证，且不能拿本地 Qwen 前缀数当命中数。")
else:
    deepseek_prefix = None
    print("尚无本实践的真实服务端对照；设置 DEEPSEEK_API_KEY 后运行 python scripts/record_context_cache_pairs.py，再执行本格。")


{'排列': 'question_first', '轮次': 1, '输入Token': 1474, '命中Token': 0, '未命中Token': 1474}
{'排列': 'question_first', '轮次': 2, '输入Token': 1473, '命中Token': 0, '未命中Token': 1473}
{'排列': 'question_first', '轮次': 3, '输入Token': 1473, '命中Token': 0, '未命中Token': 1473}
{'排列': 'stable_first', '轮次': 1, '输入Token': 1474, '命中Token': 0, '未命中Token': 1474}
{'排列': 'stable_first', '轮次': 2, '输入Token': 1473, '命中Token': 1280, '未命中Token': 193}
{'排列': 'stable_first', '轮次': 3, '输入Token': 1473, '命中Token': 1280, '未命中Token': 193}
同一排列内比较第 1、2、3 次；服务端命中不保证，且不能拿本地 Qwen 前缀数当命中数。


## 5. 练习：先预测，再看可复查的数字

先把本地 Qwen 两种排列的共同前缀相减，再从 DeepSeek 六次请求中找到每种排列的第三次命中数。它们能否证明回答质量提高？本次服务端记录能否证明所有将来请求都会命中？先写下判断，再运行下一格。


In [7]:
student_layout = "stable_first"  # TODO：改为 question_first，再观察前缀长度。
student = layouts[student_layout]
print("选择:", student_layout)
print("共同前缀 Token:", student["shared_prefix_tokens"])
print("本次实际回答:", student["answer_a"])
if cache_rows:
    third = next(row for row in cache_rows if row["排列"] == student_layout and row["轮次"] == 3)
    print("DeepSeek 同排列第 3 次命中 Token:", third["命中Token"])
else:
    print("尚无 DeepSeek 对照记录；不推断服务端命中。")


选择: stable_first
共同前缀 Token: 52
本次实际回答: progress.json
DeepSeek 同排列第 3 次命中 Token: 1280


## 6. 参考分析

本地 Qwen 实验里，稳定资料在前的共同前缀较长，两种回答在这次相同。DeepSeek 对照需单独看六条服务用量：只在本次记录中比较各排列从第一次到第三次的变化。它们使用不同的模型和 Prompt，不能把 Qwen 前缀长度当成 DeepSeek 命中数，也不能从一次服务端试验推出稳定的费用或质量优势。


In [8]:
stable = layouts["stable_first"]["shared_prefix_tokens"]
question_first = layouts["question_first"]["shared_prefix_tokens"]
print("共同前缀差:", stable - question_first, "Token")
print("两种回答相同:", layouts["stable_first"]["answer_a"] == layouts["question_first"]["answer_a"])
assert stable > question_first
assert layouts["stable_first"]["answer_a"] == layouts["question_first"]["answer_a"]
if cache_rows:
    third_hits = {row["排列"]: row["命中Token"] for row in cache_rows if row["轮次"] == 3}
    print("DeepSeek 第 3 次命中 Token:", third_hits)
    print("这是本次服务返回的用量，不是未来每次请求的保证。")


共同前缀差: 24 Token
两种回答相同: True
DeepSeek 第 3 次命中 Token: {'question_first': 0, 'stable_first': 1280}
这是本次服务返回的用量，不是未来每次请求的保证。


## 7. 交付：KV 与 Prompt 排列审计报告

把模型身份、真实 K/V 张量、两种排列的共同前缀和实际回答，以及本实践六次 DeepSeek 请求的服务端缓存字段保存为结构化记录。两组实验模型不同，报告分别注明来源；下次改变模型、精度、资料或 Prompt 时应重新运行。


In [9]:
RUN_DIR = ROOT / ".local/student-runs/l04"
RUN_DIR.mkdir(parents=True, exist_ok=True)
audit = {
    "local_model": {"model": record["model"], "revision": record["revision"],
                    "device": record["device"], "dtype": record["dtype"]},
    "kv": kv,
    "prompt_layout": {
        name: {"shared_prefix_tokens": item["shared_prefix_tokens"],
               "answer_a": item["answer_a"]}
        for name, item in layouts.items()
    },
    "provider_usage_source": {
        "status": "real_record" if deepseek_prefix else "not_recorded",
        "path": "04 上下文工程与记忆/数据/实验记录/deepseek-prefix-cache.json",
        "model": deepseek_prefix["model"] if deepseek_prefix else None,
        "recorded_at": deepseek_prefix["recorded_at"] if deepseek_prefix else None,
        "rows": cache_rows,
    },
    "interpretation": "本地共同前缀不等于服务端缓存命中；单次回答不代表整体质量。",
}
audit_path = RUN_DIR / "kv-prompt-audit.json"
audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
assert json.loads(audit_path.read_text(encoding="utf-8"))["kv"] == kv
print("审计报告:", audit_path.relative_to(ROOT))


审计报告: .local\student-runs\l04\kv-prompt-audit.json


## 本次小结

这本 Notebook 的真实模型前向说明：已处理 Token 的 K/V 能在增量解码中复用，计算结果与完整前向近似一致；缓存张量会随长度和并发占用内存。两种相同内容的 Prompt 排列给出不同的共同前缀，但本次回答相同。跨请求 Prompt Cache 必须看服务端字段；它和 KV Cache、学习记忆承担不同工作。

课后提交审计 JSON，并用自己的话解释“缓存的是哪段计算”“什么信息仍需显式放入本轮输入”。参阅 [Transformers 缓存机制](https://huggingface.co/docs/transformers/v4.56.2/en/cache_explanation)、[Qwen2.5-0.5B-Instruct 模型卡](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)。
